# We create a dataframe of new fixed image data of embryos with the nuclei pre-segmented and p cell identified.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import os
# import json
%matplotlib inline

In [ ]:
np.version.version

# Read the nstats files from folder Ncoords_1_txt

### Implementation note: after reading all files, they are put into a dataframe (df_reduced) in a sequential order based on the order in which the folders were accessed.

In [ ]:
working_path = 'Ncoords_1_txt//'

In [ ]:
entries = os.listdir(working_path)

In [ ]:
# check that num entries == 4*number of embryos in dataset
%whos

In [ ]:
# nstats dataframe per experiment - takes ~1h for 42 embryos
df_nstats = []
for file_name in entries:
    # print(file_name)
    if file_name.startswith('nstats')==True:
        # print(file_name)
        # print(file_name[0:-4])
        df_name = ''.join(file_name[0:-4])
        full_name = working_path[0:-1]+df_name+'.txt'
        #df_name_new = ''.join(working_path[0:-1]+file_name[6:-4])
        df_name_new = ''.join(file_name[6:-4])
        print(df_name_new)
#     pd.read_csv("../data_folder/data.csv")
        temp_df = pd.read_csv(full_name, delimiter=',')
        temp_df.insert(1, "sample_id", df_name_new, True)
        df_nstats.append(temp_df)

In [ ]:
# check for correct number of samples
len(df_nstats)

In [ ]:
keep_columns = ['Volume', 'sample_id', 'Centroid_1', 'Centroid_2', 'Centroid_3', 'ConvexVolume', 'EquivDiameter']

In [ ]:
df_nstats[0].columns

In [ ]:
from copy import copy, deepcopy

working_path = 'Ncoords_1_txt//'
entries = os.listdir(working_path)

df_reduced = []
count = 1
for df_temp in df_nstats:
#     print(count)
    count += 1
    df_t = deepcopy(df_temp)
    df_t = df_t[keep_columns]
#     print(df_temp.loc[0, 'sample_id'][6:])
#     print('name')
#     print('pcellnucID'+df_temp.loc[0, 'sample_id'][6:] + '.txt')
    for file_name in entries:
#         print(file_name)
#         if df_temp.loc[0, 'sample_id'][6:] == 'Sample17FOV1':
#             print(file_name)
        if file_name == 'pcellMergednucID' + df_temp.loc[0, 'sample_id'][:] + '.txt':
#             print('here', df_temp.loc[0, 'sample_id'][6:])
            
            temp_name = working_path[0:-1]+'pcellMergednucID'+df_temp.loc[0, 'sample_id'][:] + '.txt'
            csv_name = 'dataset1_'+df_temp.loc[0, 'sample_id'][:]+'.csv'
#         if file_name.startswith('pcellnucIDSample')==True and temp_name == :
# print(temp_name)
            df_pcell = pd.read_csv(temp_name, delimiter=',', header = None)
            df_t['Pcell_id'] = df_pcell.loc[0,0].item()
    df_t["cell_name"] = df_t.index + 1
    df_t["Z2/Z3"] = 0

    current_sample_id = df_temp.loc[0, "sample_id"]
    
    if current_sample_id.startswith("_separate"):
        base_name = current_sample_id.replace("_separate", "", 1)

        z2_file = working_path[0:-1] + 'z2cellnucID' + base_name + '.txt'
        z3_file = working_path[0:-1] + 'z3cellnucID' + base_name + '.txt'

        z_ids = []

        if os.path.exists(z2_file):
            z2_id = pd.read_csv(z2_file, header=None).iloc[0, 0]
            z_ids.append(int(z2_id))

        if os.path.exists(z3_file):
            z3_id = pd.read_csv(z3_file, header=None).iloc[0, 0]
            z_ids.append(int(z3_id))

        # mark Z2 and Z3 as 1
        df_t.loc[df_t["cell_name"].isin(z_ids), "Z2/Z3"] = 1

    df_reduced.append(df_t)
    # Save
#    df_t.to_csv(csv_name, index=False)

In [ ]:
file_name

In [ ]:
df_temp.loc[0, 'sample_id'][:]

In [ ]:
# Save
# np.save('df_reduced_1.npy', df_reduced)

In [ ]:
df_reduced[0]

In [ ]:
# Load
# test_df = pd.read_csv('test.csv', names=keep_columns)

In [ ]:
# check for correct number of samples
len(df_reduced)

In [ ]:
# Rename columns.
for i in df_reduced:
    i.rename(columns = {'Centroid_1':'X'}, inplace = True)
    i.rename(columns = {'Centroid_2':'Y'}, inplace = True)
    i.rename(columns = {'Centroid_3':'Z'}, inplace = True)

In [ ]:
df_reduced[0]

In [ ]:
# Save
#np.save('df_reduced.npy', df_reduced)
np.save('df_reduced.npy', np.array(df_reduced, dtype=object), allow_pickle=True)

In [ ]:
# Load
df_reduced = np.load('df_reduced.npy', allow_pickle=True)

In [ ]:
df_reduced[5]

In [ ]:
with pd.option_context('display.max_rows', 200):
    display(df_reduced[0])